# 02 - Auto-Categorization Model

Train a Keras text classifier to predict transaction category from merchant name.

**Architecture**: Embedding → LSTM(32) → Dense(64) → Softmax(12)

**Pipeline**:
1. Tokenize merchant names
2. Train LSTM classifier
3. Evaluate (accuracy, F1, confusion matrix)
4. Export to TFLite with INT8 quantization
5. Export tokenizer + category labels for Flutter

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

from preprocessing import APP_CATEGORIES, CATEGORY_TO_IDX, IDX_TO_CATEGORY
from evaluate import print_classification_metrics, plot_confusion_matrix, plot_training_history
from export_tflite import (
    export_keras_to_tflite, export_tokenizer,
    export_category_labels, verify_tflite_model
)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 1. Load Preprocessed Data

In [ ]:
df = pd.read_csv('../data/processed/categorization_train.csv')
print(f'Dataset: {len(df)} rows, {df["category"].nunique()} categories')
print(f'\nCategory distribution:')
print(df['category'].value_counts().sort_index())

texts = df['merchant_clean'].values
labels = df['category_idx'].values
print(f'\nLabel range: {labels.min()} to {labels.max()}')

## 2. Tokenization

In [ ]:
# Hyperparameters
VOCAB_SIZE = 2000
MAX_LEN = 10      # max tokens per merchant name
EMBEDDING_DIM = 32
LSTM_UNITS = 32
NUM_CLASSES = len(APP_CATEGORIES)

# Tokenize
tokenizer = tf.keras.preprocessing.text.Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token='<OOV>',
)
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
padded = tf.keras.preprocessing.sequence.pad_sequences(
    sequences, maxlen=MAX_LEN, padding='post', truncating='post'
)

print(f'Vocabulary size: {len(tokenizer.word_index)} unique tokens')
print(f'Using top {VOCAB_SIZE} tokens')
print(f'Padded shape: {padded.shape}')
print(f'\nSample tokenization:')
for i in range(5):
    print(f'  "{texts[i]}" → {padded[i].tolist()}')

## 3. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    padded, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')
print(f'\nTrain label distribution:')
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {IDX_TO_CATEGORY[u]:15s}: {c}')

## 4. Build Model

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LEN),
    tf.keras.layers.LSTM(LSTM_UNITS),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

## 5. Train Model

In [ ]:
# Compute class weights to handle imbalance
from sklearn.utils.class_weight import compute_class_weight

class_weights_arr = compute_class_weight(
    'balanced', classes=np.unique(y_train), y=y_train
)
class_weights = dict(enumerate(class_weights_arr))
print('Class weights:', {IDX_TO_CATEGORY[k]: round(v, 2) for k, v in class_weights.items()})

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=30,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=5, restore_best_weights=True, monitor='val_accuracy'
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            factor=0.5, patience=3, monitor='val_loss'
        ),
    ],
)

In [ ]:
# Plot training curves
fig = plot_training_history(history, metrics=('accuracy', 'loss'))
fig.savefig('../models/saved/categorizer_training_curves.png', dpi=100)
plt.show()

## 6. Evaluate on Test Set

In [ ]:
import matplotlib.pyplot as plt

# Predictions
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Metrics
metrics = print_classification_metrics(y_test, y_pred, label_names=APP_CATEGORIES)

In [ ]:
# Confusion matrix
fig = plot_confusion_matrix(y_test, y_pred, label_names=APP_CATEGORIES)
fig.savefig('../models/saved/categorizer_confusion_matrix.png', dpi=100)
plt.show()

In [ ]:
# Confidence analysis
max_probs = np.max(y_pred_probs, axis=1)
correct = y_pred == y_test

print(f'Average confidence: {max_probs.mean():.3f}')
print(f'Confidence when correct:   {max_probs[correct].mean():.3f}')
print(f'Confidence when incorrect: {max_probs[~correct].mean():.3f}')
print(f'\nConfidence thresholds:')
for threshold in [0.5, 0.6, 0.7, 0.8, 0.9]:
    above = max_probs >= threshold
    if above.sum() > 0:
        acc_above = (y_pred[above] == y_test[above]).mean()
        print(f'  >= {threshold:.0%}: {above.sum()} samples ({above.mean():.1%}), accuracy {acc_above:.3f}')

## 7. Export to TFLite

In [ ]:
# Save full Keras model
model.save('../models/saved/categorizer_keras')
print('Keras model saved.')

# Export TFLite with quantization
tflite_meta = export_keras_to_tflite(
    model,
    output_path='../models/tflite/categorizer.tflite',
    quantize=True,
)
print(f'\nTFLite model: {tflite_meta["size_kb"]:.1f} KB')

In [ ]:
# Export tokenizer word_index
export_tokenizer(
    tokenizer.word_index,
    output_path='../models/tflite/tokenizer.json'
)

# Export category labels (ordered)
export_category_labels(
    APP_CATEGORIES,
    output_path='../models/tflite/categories.json'
)

# Export model config (hyperparams Flutter needs)
import json
config = {
    'vocab_size': VOCAB_SIZE,
    'max_len': MAX_LEN,
    'num_classes': NUM_CLASSES,
    'categories': APP_CATEGORIES,
    'confidence_threshold': 0.8,
    'metrics': {
        'accuracy': float(metrics['accuracy']),
        'f1_weighted': float(metrics['f1_weighted']),
    },
}
with open('../models/tflite/categorizer_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('\nModel config saved.')

In [ ]:
# Verify TFLite model produces same outputs
sample = X_test[:5].astype(np.float32)
tflite_output = verify_tflite_model('../models/tflite/categorizer.tflite', sample)

keras_output = model.predict(sample)

print(f'\nKeras vs TFLite prediction comparison:')
for i in range(5):
    keras_cat = IDX_TO_CATEGORY[np.argmax(keras_output[i])]
    tflite_cat = IDX_TO_CATEGORY[np.argmax(tflite_output[i])]
    match = 'MATCH' if keras_cat == tflite_cat else 'MISMATCH'
    print(f'  Sample {i}: Keras={keras_cat}, TFLite={tflite_cat} [{match}]')

## 8. Test with Real Merchant Names

In [ ]:
# Predict on real-world merchant names
test_merchants = [
    'Starbucks', 'Walmart', 'Uber', 'Netflix', 'Mortgage Payment',
    'Shell Gas', 'Amazon', 'CVS Pharmacy', 'Biweekly Paycheck',
    'Electric Company', 'Pizza Hut', 'Gym Membership',
]

from preprocessing import clean_merchant_text

cleaned = [clean_merchant_text(m) for m in test_merchants]
seqs = tokenizer.texts_to_sequences(cleaned)
padded_test = tf.keras.preprocessing.sequence.pad_sequences(
    seqs, maxlen=MAX_LEN, padding='post', truncating='post'
)

probs = model.predict(padded_test)

print(f'{"Merchant":25s} {"Predicted":15s} {"Confidence":>10s}  Top 3')
print('-' * 80)
for i, merchant in enumerate(test_merchants):
    top3_idx = np.argsort(probs[i])[-3:][::-1]
    pred_cat = IDX_TO_CATEGORY[top3_idx[0]]
    conf = probs[i][top3_idx[0]]
    top3 = [(IDX_TO_CATEGORY[j], f'{probs[i][j]:.2f}') for j in top3_idx]
    auto = 'AUTO' if conf >= 0.8 else 'SUGGEST'
    print(f'{merchant:25s} {pred_cat:15s} {conf:>9.1%} [{auto}]  {top3}')

## Results Summary

### Model Architecture
```
Embedding(2000, 32) → LSTM(32) → Dense(64, relu) → Dropout(0.3) → Dense(12, softmax)
```

### Files Exported for Flutter
| File | Description |
|------|-------------|
| `categorizer.tflite` | Quantized TFLite model |
| `tokenizer.json` | Word → index mapping |
| `categories.json` | Ordered category labels |
| `categorizer_config.json` | Hyperparams + metrics |

### Key Metrics
- Fill in after running: accuracy, F1 weighted, model size in KB
- Confidence threshold: 80% (auto-assign above, suggest top 3 below)

### Next: Copy exports to Flutter
```bash
cp ../models/tflite/categorizer.tflite ../../app/assets/models/
cp ../models/tflite/tokenizer.json ../../app/assets/models/
cp ../models/tflite/categories.json ../../app/assets/models/
cp ../models/tflite/categorizer_config.json ../../app/assets/models/
```